Machine learning CW.  
Housing data, Available at: https://www.zillow.com/research/data/.  
Student ID: 00017747.  

First, we import all necessary libs

In [11]:
import numpy
import pandas
import matplotlib

print('numpy', numpy.__version__)
print('pandas', pandas.__version__)
print('matplotlib', matplotlib.__version__)


numpy 2.3.4
pandas 2.3.3
matplotlib 3.10.7


#### 1. Load dataset  

Next, we load our chosen housing dataset from zillow.com using pandas's read_csv fn

In [12]:
df = pandas.read_csv("housing_data.csv")
df.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,2000-01-31,2000-02-29,2000-03-31,2000-04-30,2000-05-31,...,2024-12-31,2025-01-31,2025-02-28,2025-03-31,2025-04-30,2025-05-31,2025-06-30,2025-07-31,2025-08-31,2025-09-30
0,102001,0,United States,country,NaN,123328.165417,123545.139196,123814.218630,124391.341197,125055.540080,...,365372.217349,366007.694288,366471.281294,366193.660206,365661.222907,364970.033287,364347.749809,363917.639079,363688.149053,363931.687425
1,394913,1,"New York, NY",msa,NY,222096.674116,223040.459669,223992.986378,225923.174072,227921.950272,...,696533.300128,697633.014562,698948.623497,700676.336950,703154.623667,704955.084297,706457.122100,707650.653457,708454.797763,709880.464236
2,753899,2,"Los Angeles, CA",msa,CA,222620.175787,223448.605299,224552.065508,226747.580645,229148.785908,...,968173.703124,968714.163371,966675.169598,961554.657443,957217.264182,952421.630548,948047.015396,945442.656758,944366.276929,945428.022538
3,394463,3,"Chicago, IL",msa,IL,155857.310945,156001.589412,156276.370294,156959.956841,157782.229234,...,332649.038316,334015.386374,335383.265039,336242.517133,336852.936258,337148.112023,337514.741114,338367.259828,339378.680707,340732.680569
4,394514,4,"Dallas, TX",msa,TX,128023.642757,128080.664935,128146.217766,128316.451582,128540.900207,...,377150.293426,376619.688458,375829.848426,374322.685671,372189.097900,369765.587859,367394.345281,365371.532830,364005.278732,363356.229199


In [13]:
print('original shape', df.shape) 

target_state = 'NY'
df_subset = df[df['StateName']==target_state].copy()
print(f"subset shape (state: {target_state}):", df_subset.shape)

original shape (895, 314)
subset shape (state: NY): (26, 314)


next melting

In [15]:
id_vars = ['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName']
date_columns = [c for c in df_subset.columns if c not in id_vars]

df_melted = df_subset.melt(
    id_vars=id_vars, 
    value_vars=date_columns, 
    var_name='Date', 
    value_name='Price'
)

df_melted['Date'] = pandas.to_datetime(df_melted['Date'])
df_melted = df_melted.sort_values(by=['RegionName', 'Date'])
df_clean = df_melted.dropna(subset=['Price'])   
print("Shape after Melting and Cleaning:", df_clean.shape)
print(df_clean.head())

Shape after Melting and Cleaning: (8019, 7)
     RegionID  SizeRank  RegionName RegionType StateName       Date  \
3      394308        64  Albany, NY        msa        NY 2000-01-31   
29     394308        64  Albany, NY        msa        NY 2000-02-29   
55     394308        64  Albany, NY        msa        NY 2000-03-31   
81     394308        64  Albany, NY        msa        NY 2000-04-30   
107    394308        64  Albany, NY        msa        NY 2000-05-31   

             Price  
3    119117.984199  
29   119505.438260  
55   119758.918109  
81   120361.673989  
107  120890.242210  


c

In [ ]:
df_clean['Year'] = df_clean['Date'].dt.year
df_clean['Month'] = df_clean['Date'].dt.month
df_clean['Quarter'] = df_clean['Date'].dt.quarter

df_model = df_clean.groupby('RegionName').apply(lambda group: group.assign(
    Price_Next_Month = group['Price'].shift(-1), 
    Price_Lag_1M = group['Price'].shift(1),
    Price_Lag_6M = group['Price'].shift(6),
    Price_Lag_12M = group['Price'].shift(12),
    Price_Roll_3M = group['Price'].rolling(window=3, min_periods=1).mean().shift(1)
    
)).reset_index(drop=True)

df_model = df_model.dropna(subset=['Price_Next_Month', 'Price_Lag_12M', 'Price_Roll_3M'])

print("Shape after Feature Engineering and Final Cleaning:", df_model.shape)
print(df_model[['RegionName', 'Date', 'Price', 'Price_Next_Month', 'Price_Lag_1M']].head(15))

Shape after Feature Engineering and Final Cleaning: (7681, 15)
    RegionName       Date          Price  Price_Next_Month   Price_Lag_1M
12  Albany, NY 2001-01-31  124052.764500     124323.494035  123800.167646
13  Albany, NY 2001-02-28  124323.494035     124497.391840  124052.764500
14  Albany, NY 2001-03-31  124497.391840     125027.760682  124323.494035
15  Albany, NY 2001-04-30  125027.760682     125380.558464  124497.391840
16  Albany, NY 2001-05-31  125380.558464     125994.385667  125027.760682
17  Albany, NY 2001-06-30  125994.385667     126504.489255  125380.558464
18  Albany, NY 2001-07-31  126504.489255     127174.990721  125994.385667
19  Albany, NY 2001-08-31  127174.990721     127658.440628  126504.489255
20  Albany, NY 2001-09-30  127658.440628     128325.575745  127174.990721
21  Albany, NY 2001-10-31  128325.575745     129053.367493  127658.440628
22  Albany, NY 2001-11-30  129053.367493     129845.266289  128325.575745
23  Albany, NY 2001-12-31  129845.266289     1305

/var/folders/lh/c7rm8s2j7bg03wh7jwz5q_sm0000gn/T/ipykernel_83404/468336580.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Year'] = df_clean['Date'].dt.year
/var/folders/lh/c7rm8s2j7bg03wh7jwz5q_sm0000gn/T/ipykernel_83404/468336580.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Month'] = df_clean['Date'].dt.month
/var/folders/lh/c7rm8s2j7bg03wh7jwz5q_sm0000gn/T/ipykernel_83404/468336580.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFr

In [ ]:
X = df_model[[
    'RegionID', 'SizeRank', 'Year', 'Month', 'Quarter', 
    'Price_Lag_1M', 'Price_Lag_6M', 'Price_Lag_12M', 'Price_Roll_3M'
]]
Y = df_model['Price_Next_Month']

SPLIT_DATE = pandas.to_datetime('2023-01-01') 

train_mask = df_model['Date'] < SPLIT_DATE
test_mask = df_model['Date'] >= SPLIT_DATE

X_train, X_test = X[train_mask], X[test_mask]
Y_train, Y_test = Y[train_mask], Y[test_mask]

print(f"Training set size: {len(X_train)} rows")
print(f"Testing set size: {len(X_test)} rows")

Training set size: 6849 rows
Testing set size: 832 rows
